# 9. Introduction to Natural Language Processing

**Machine Learning Fundamentals and Predictive Analytics — Notebook 9 of 11**

Notebook 8 fed word counts into Naive Bayes without asking where they came from. This notebook
answers that question: how does raw text become numbers a model can use at all?

We cover the classical NLP pipeline — the one that still runs in production behind search bars,
spam filters and support-ticket routers, and the one that every modern language model was built
on top of.

### What you will learn

1. **Text preprocessing**: tokenisation, stopwords, stemming vs lemmatisation
2. **Bag-of-words** and **N-grams**
3. **TF-IDF**: why raw counts mislead, and the fix
4. Classic NLP tasks: **NER**, **POS tagging**
5. **Word embeddings**: word2vec and the distributional hypothesis
6. From bag-of-words to embeddings: a head-to-head
7. **Topic modelling** with NMF and LDA
8. A complete **sentiment analysis** pipeline
9. Where classical NLP ends and modern LLMs begin

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.dummy import DummyClassifier

rng = np.random.default_rng(seed=9)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)

---
## 9.1 Text preprocessing

Raw text is messy: inconsistent case, punctuation, plurals, common words that carry no topic
information. Preprocessing normalises it before it ever reaches a model.

**Tokenisation** — splitting text into units (usually words). Looks trivial, is not: "don't"
is one token or two depending on the tokeniser; "New York" might be one entity or two words.

**Lowercasing** — "Apple" the company and "apple" the fruit collide, which is usually an
acceptable cost for the vocabulary reduction it buys.

**Stopword removal** — drop high-frequency, low-information words ("the", "is", "and"). Cuts
vocabulary size and noise. Keep them if you care about grammar or style (authorship, sentiment
nuance — "not good" loses its meaning if "not" is removed as a stopword).

**Stemming vs lemmatisation** — both reduce words to a common form, but differently:

| | Method | "running" → | "better" → | "was" → |
|---|---|---|---|---|
| **Stemming** | Chops suffixes with rules (fast, crude) | "run" | "better" | "wa" |
| **Lemmatisation** | Dictionary + grammar lookup (slower, correct) | "run" | "good" | "be" |

Stemming can produce non-words; lemmatisation always produces valid dictionary forms but needs
part-of-speech context to do it well.

In [ ]:
sample = "The runners were running faster than the other running teams had ever run before."

def simple_tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

tokens = simple_tokenize(sample)
print(f"Original : {sample}")
print(f"Tokens   : {tokens}\n")

STOPWORDS = {"the", "was", "were", "than", "other", "had", "ever", "before", "a", "an",
            "is", "are", "to", "of", "in", "on", "and", "or"}
no_stop = [t for t in tokens if t not in STOPWORDS]
print(f"After stopword removal ({len(tokens)} -> {len(no_stop)} tokens):")
print(f"  {no_stop}")

In [ ]:
# A tiny rule-based stemmer, to show the mechanism (nltk's PorterStemmer does this properly)
def toy_stem(word):
    if word.endswith("ing") and len(word) > 5:
        return word[:-3]
    if word.endswith("ers") and len(word) > 5:
        return word[:-3]
    if word.endswith("s") and not word.endswith("ss") and len(word) > 3:
        return word[:-1]
    return word

lemma_lookup = {"running": "run", "runners": "runner", "faster": "fast", "run": "run",
                "teams": "team"}

print(f"{'word':<12}{'toy stem':<12}{'lemma (dict lookup)':<20}")
for w in ["running", "runners", "faster", "run", "teams"]:
    print(f"{w:<12}{toy_stem(w):<12}{lemma_lookup.get(w, w):<20}")

print("\nA real stemmer (Porter's algorithm) applies about 60 rewrite rules in sequence.")
print("A real lemmatiser needs a dictionary AND the word's part of speech: 'saw' lemmatises")
print("to 'see' as a verb but stays 'saw' as a noun (the tool).")
print("\nRule of thumb: stemming for speed and search-engine-style recall; lemmatisation")
print("when the output text will be read by a human, or fed to something that needs valid")
print("words (like a downstream lemma-based lexicon).")

---
## 9.2 Bag-of-words and N-grams

The **bag-of-words** model represents a document as a vector of word counts, discarding order
entirely. "dog bites man" and "man bites dog" become identical vectors — an obvious and
important limitation.

**N-grams** partially restore order by treating consecutive word sequences as tokens:

- **Unigrams** (n=1): "dog", "bites", "man"
- **Bigrams** (n=2): "dog bites", "bites man"
- **Trigrams** (n=3): "dog bites man"

Bigrams capture negation ("not good" vs "good"), fixed phrases, and simple word order, at the
cost of a much larger, sparser vocabulary.

In [ ]:
corpus = ["the dog bites the man", "the man bites the dog", "the dog runs fast"]

cv1 = CountVectorizer(ngram_range=(1, 1))
X1 = cv1.fit_transform(corpus)
print("UNIGRAMS -- 'dog bites man' and 'man bites dog' become IDENTICAL:")
print(pd.DataFrame(X1.toarray(), columns=cv1.get_feature_names_out(),
                   index=[f"doc{i}" for i in range(3)]))

cv2 = CountVectorizer(ngram_range=(1, 2))
X2 = cv2.fit_transform(corpus)
print(f"\nUNIGRAMS + BIGRAMS -- now they differ ({X1.shape[1]} -> {X2.shape[1]} features):")
print(pd.DataFrame(X2.toarray(), columns=cv2.get_feature_names_out(),
                   index=[f"doc{i}" for i in range(3)]).to_string())
print(f"\nCosine similarity of doc0 and doc1:")
print(f"  unigrams        : {cosine_similarity(X1[0], X1[1])[0,0]:.4f}  <- looks identical")
print(f"  unigrams+bigrams: {cosine_similarity(X2[0], X2[1])[0,0]:.4f}  <- correctly different")

In [ ]:
# Negation, a concrete case where bigrams matter for sentiment
neg_corpus = ["this movie was good", "this movie was not good",
             "the food was great", "the food was not great at all"]
for n_range, label in [((1, 1), "unigrams"), ((1, 2), "uni+bigrams")]:
    cv = CountVectorizer(ngram_range=n_range)
    X = cv.fit_transform(neg_corpus)
    print(f"\n{label}: sim('good' vs 'not good') = "
          f"{cosine_similarity(X[0], X[1])[0,0]:.4f}")
print("\nWith unigrams alone, 'not good' and 'good' share every word except 'not' and look")
print("nearly identical. Bigrams give the model the token 'not good' directly.")

# Vocabulary size explodes with n -- the practical cost
big_corpus = [" ".join(rng.choice(list("abcdefghij"), 12)) for _ in range(200)]
for n in (1, 2, 3):
    v = CountVectorizer(ngram_range=(1, n)).fit(big_corpus)
    print(f"  ngram_range=(1,{n}): vocabulary size = {len(v.vocabulary_)}")
print("\nThis is why min_df / max_features are almost always set alongside ngram_range.")

---
## 9.3 TF-IDF: why raw counts mislead

Raw word counts have a problem: common words ("the", "is") dominate every document's vector
even though they carry no information about what makes a document distinctive. **TF-IDF**
(term frequency – inverse document frequency) downweights words that appear in *most*
documents and upweights words that are rare across the corpus but frequent in a particular one.

$$\text{tf-idf}(t, d) = \text{tf}(t,d) \times \text{idf}(t), \qquad
\text{idf}(t) = \log\frac{1+n}{1+\text{df}(t)} + 1$$

- $\text{tf}(t,d)$ — how often term $t$ appears in document $d$
- $\text{df}(t)$ — how many documents contain $t$ at all
- A word in **every** document gets $\text{idf} \approx 1$ (barely counted)
- A word in **one** document out of many gets a large idf (heavily weighted)

scikit-learn also **L2-normalises** each document vector by default, so document length does
not distort similarity.

In [ ]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "quantum entanglement puzzles physicists",
]
cvec = CountVectorizer()
counts = cvec.fit_transform(docs).toarray()
tfidf_vec = TfidfVectorizer()
tfidf = tfidf_vec.fit_transform(docs).toarray()

vocab = cvec.get_feature_names_out()
print("Raw counts:")
print(pd.DataFrame(counts, columns=vocab, index=[f"doc{i}" for i in range(3)]))
print("\nTF-IDF:")
print(pd.DataFrame(tfidf.round(3), columns=vocab, index=[f"doc{i}" for i in range(3)]))

idf = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out()).sort_values()
print("\nIDF values, smallest (most common) to largest (most distinctive):")
print(idf.round(3).to_string())
print("\n'the' appears in every document -> lowest idf, nearly ignored.")
print("'quantum', 'entanglement', 'physicists' appear in one document only -> highest idf.")

In [ ]:
# TF-IDF vs raw counts as classifier inputs: the empirical payoff
spam_words = ["free", "win", "click", "now", "urgent", "prize", "claim", "money", "offer"]
ham_words = ["meeting", "report", "project", "deadline", "review", "team", "client",
             "schedule", "notes"]
filler = ["the", "a", "to", "and", "of", "for", "your", "you", "is"]

def make_msg(spam, g):
    topic = spam_words if spam else ham_words
    words = list(g.choice(topic, g.integers(2, 5))) + list(g.choice(filler, g.integers(4, 9)))
    g.shuffle(words)
    return " ".join(words)

y_msg = (rng.random(1200) < 0.4).astype(int)
X_msg = [make_msg(bool(s), rng) for s in y_msg]
Xa, Xb, ya, yb = train_test_split(X_msg, y_msg, test_size=0.3, random_state=0, stratify=y_msg)

for name, vec in [("CountVectorizer", CountVectorizer()), ("TfidfVectorizer", TfidfVectorizer())]:
    pipe = make_pipeline(vec, LogisticRegression(max_iter=2000))
    cvv = cross_val_score(pipe, Xa, ya, cv=SKF).mean()
    print(f"  {name:<18} CV accuracy {cvv:.4f}")
print("\nOn balanced, filler-heavy synthetic text the gap is modest -- TF-IDF earns its")
print("keep most clearly on real corpora with long documents and skewed word frequencies")
print("(a handful of function words dominating raw counts).")

---
## 9.4 Named Entity Recognition and POS tagging

Two classic **sequence labelling** tasks — assigning a tag to every token, not the whole
document.

**Part-of-speech (POS) tagging** labels each word's grammatical role: noun, verb, adjective...
**Named Entity Recognition (NER)** labels spans that refer to real-world entities: person,
organisation, location, date.

Both were historically solved with **Hidden Markov Models** or **Conditional Random Fields** —
sequence models that use the tags of neighbouring words as features, because "bank" is a noun
after "the" and a verb after "will". Today, production systems use pretrained neural taggers
(spaCy, Stanza), but the *task definition* is unchanged, and the classical approach is worth
seeing once because it makes the sequential dependency explicit.

In [ ]:
# A minimal rule-based POS tagger, to show the idea without extra dependencies
DETERMINERS = {"the", "a", "an", "this", "that"}
PRONOUNS = {"i", "you", "he", "she", "it", "we", "they"}
PREPOSITIONS = {"in", "on", "at", "by", "with", "to", "of", "for", "from"}
VERB_SUFFIXES = ("ing", "ed", "s")
NOUN_SUFFIXES = ("tion", "ness", "ment", "ity")

def rule_based_pos(tokens):
    tags = []
    for i, t in enumerate(tokens):
        tl = t.lower()
        if tl in DETERMINERS:
            tags.append("DET")
        elif tl in PRONOUNS:
            tags.append("PRON")
        elif tl in PREPOSITIONS:
            tags.append("PREP")
        elif tl.endswith(NOUN_SUFFIXES):
            tags.append("NOUN")
        elif i > 0 and tags[-1] == "DET":
            tags.append("NOUN")                          # after a determiner: usually a noun
        elif tl.endswith(VERB_SUFFIXES) and tags[-1:] != ["DET"]:
            tags.append("VERB")
        else:
            tags.append("NOUN" if t[0].isupper() else "OTHER")
    return tags

sentence = "The quick brown fox jumps over the lazy dog".split()
tags = rule_based_pos(sentence)
print(pd.DataFrame({"token": sentence, "POS (rule-based)": tags}).to_string(index=False))
print("\nA real tagger (e.g. spaCy) would also use the WORD IDENTITY (a huge lexicon),")
print("suffix statistics learned from a treebank, and the tags of BOTH neighbours --")
print("this toy only looks one step back, which is why it mislabels 'quick' and 'brown'.")

In [ ]:
# A minimal gazetteer-based NER, to show the mechanism
PERSON_NAMES = {"john", "sarah", "priya", "wei", "carlos"}
ORG_SUFFIXES = ("inc", "corp", "ltd", "llc")
LOCATIONS = {"chennai", "london", "tokyo", "paris", "mumbai"}

def rule_based_ner(text):
    tokens = text.split()
    entities = []
    for t in tokens:
        clean = t.strip(",.")
        low = clean.lower()
        if low in PERSON_NAMES:
            entities.append((clean, "PERSON"))
        elif low in LOCATIONS:
            entities.append((clean, "LOCATION"))
        elif low.endswith(ORG_SUFFIXES):
            entities.append((clean, "ORGANIZATION"))
    return entities

text = "John met Sarah in Chennai to discuss the Acme Corp contract."
print(f"Text: {text}\n")
print("Gazetteer matches:")
for tok, ent in rule_based_ner(text):
    print(f"  {tok:<10} -> {ent}")
print("\nThis catches names IN THE DICTIONARY and nothing else -- it cannot recognise")
print("'Priyanka' or 'Bengaluru' if they are not listed, and it cannot use CONTEXT the way")
print("a trained model does ('Amazon' the company vs 'Amazon' the river). Modern NER models")
print("are trained sequence classifiers (CRF, BiLSTM-CRF, or transformer-based) that learn")
print("these distinctions from thousands of labelled sentences instead of a fixed list.")

---
## 9.5 Word embeddings

Bag-of-words treats every word as an independent, meaningless index — "excellent" and
"outstanding" are as unrelated as "excellent" and "bicycle". **Word embeddings** fix this by
learning a dense vector for each word such that similar words end up close together.

### The distributional hypothesis

*"You shall know a word by the company it keeps"* (Firth, 1957). Words that appear in similar
contexts tend to have similar meanings. **word2vec** operationalises this directly: train a
tiny neural network to predict a word from its neighbours (CBOW) or neighbours from a word
(Skip-gram). The learned *weights*, not the predictions, are the embeddings — the task is a
pretext for learning the representation.

The famous result: the geometry of the embedding space encodes analogies —

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

We do not have a pretrained word2vec model bundled with scikit-learn, so we **build a tiny toy
embedding by hand** with the same mechanism — co-occurrence statistics reduced with SVD — to
see the geometry emerge from a small corpus.

In [ ]:
# A tiny corpus with enough co-occurrence structure to show the effect
sentences = [
    "the king rules the kingdom", "the queen rules the kingdom",
    "the king is a powerful man", "the queen is a powerful woman",
    "the prince is the son of the king", "the princess is the daughter of the queen",
    "the man walked into the palace", "the woman walked into the palace",
    "the king and the queen ruled together", "a man and a woman met the king",
    "the dog ran in the park", "the cat ran in the park",
    "the dog chased the cat", "a dog is a loyal animal",
    "a cat is an independent animal", "the park has many dogs and cats",
]

# Build a word co-occurrence matrix (window = 2) -- the classical precursor to word2vec
window = 2
vocab_words = sorted({w for s in sentences for w in s.split()} - {"the", "a", "is", "of", "an"})
w2i = {w: i for i, w in enumerate(vocab_words)}
co = np.zeros((len(vocab_words), len(vocab_words)))
for s in sentences:
    toks = [w for w in s.split() if w in w2i]
    for i, w in enumerate(toks):
        for j in range(max(0, i-window), min(len(toks), i+window+1)):
            if i != j:
                co[w2i[w], w2i[toks[j]]] += 1

print(f"Vocabulary: {vocab_words}")
print(f"Co-occurrence matrix shape: {co.shape}")

In [ ]:
# Reduce with SVD to get low-dimensional embeddings -- the same trick behind GloVe
svd = TruncatedSVD(n_components=8, random_state=0)
embeddings = svd.fit_transform(co + 1e-6)
emb_df = pd.DataFrame(embeddings, index=vocab_words)

def word_sim(w1, w2):
    return cosine_similarity([embeddings[w2i[w1]]], [embeddings[w2i[w2]]])[0, 0]

pairs = [("king", "queen"), ("king", "prince"), ("man", "woman"), ("dog", "cat"),
        ("king", "dog"), ("queen", "cat"), ("king", "cat")]
print(f"{'pair':<18}{'cosine similarity':>18}")
for a, b in pairs:
    print(f"{a+' / '+b:<18}{word_sim(a, b):>18.4f}")

print("\nRoyal words cluster with each other; animal words cluster with each other;")
print("cross-cluster pairs (king/dog) score lower -- purely from co-occurrence counts,")
print("with no labels and no meaning provided to the algorithm.")

In [ ]:
# Visualise the embedding space in 2-D
svd2 = TruncatedSVD(n_components=2, random_state=0)
emb2 = svd2.fit_transform(co + 1e-6)

plt.figure(figsize=(8, 6))
royal = {"king", "queen", "prince", "princess", "palace"}
animal = {"dog", "cat", "animal"}
for w, (x, y) in zip(vocab_words, emb2):
    colour = "crimson" if w in royal else ("steelblue" if w in animal else "grey")
    plt.scatter(x, y, color=colour, s=60)
    plt.annotate(w, (x, y), fontsize=9, xytext=(4, 4), textcoords="offset points")
plt.title("A toy embedding space learned from 16 sentences")
plt.xlabel("component 1"); plt.ylabel("component 2")
plt.show()

print("Real word2vec / GloVe embeddings do the same thing at a vastly larger scale --")
print("billions of words, 100-300 dimensions, and famous analogy arithmetic that emerges")
print("from nothing but co-occurrence statistics and a training objective.")

In [ ]:
# Document vectors from averaged word embeddings, vs bag-of-words
def doc_vector(sentence, embeddings, w2i):
    words = [w for w in sentence.split() if w in w2i]
    if not words:
        return np.zeros(embeddings.shape[1])
    return embeddings[[w2i[w] for w in words]].mean(axis=0)

test_docs = ["the king rules", "the queen rules", "the dog ran"]
doc_vecs = np.array([doc_vector(d, embeddings, w2i) for d in test_docs])

print("Document similarity from AVERAGED WORD EMBEDDINGS:")
sims_emb = cosine_similarity(doc_vecs)
print(pd.DataFrame(sims_emb.round(3), index=test_docs, columns=test_docs))

bow = CountVectorizer(vocabulary=vocab_words).fit_transform(test_docs).toarray()
print("\nDocument similarity from BAG-OF-WORDS:")
print(pd.DataFrame(cosine_similarity(bow).round(3), index=test_docs, columns=test_docs))

print("\nEmbeddings correctly rate 'king rules' and 'queen rules' as more similar to each")
print("other than either is to 'dog ran' -- even though 'king' and 'queen' share NO")
print("characters and (with 'the' as a stopword) share no words at all. Bag-of-words")
print("cannot see that; embeddings can, because the WORDS themselves are already similar.")

---
## 9.6 Topic modelling: NMF and LDA

**Topic modelling** discovers latent themes in a corpus with no labels at all — a form of
clustering, but over words instead of over data points, producing soft, overlapping groups
called **topics**.

**NMF (Non-negative Matrix Factorisation)** — factorise the TF-IDF matrix
$V \approx W H$ where $W$ (documents × topics) and $H$ (topics × words) are constrained to be
non-negative. Non-negativity is what makes the topics interpretable as "parts": a document is
a non-negative combination of topics, a topic is a non-negative combination of words.

**LDA (Latent Dirichlet Allocation)** — a generative probabilistic model: each document is a
mixture of topics, each topic is a distribution over words, both drawn from Dirichlet priors.
Works on raw counts, not TF-IDF (it needs actual counts to model as draws from a multinomial).

Both need you to choose the number of topics $k$ up front — the same problem as choosing $k$
for K-Means (statistics-adjacent Notebook 5 of this module).

In [ ]:
# A corpus mixing three latent topics, unlabelled
sport_words = ["match", "goal", "team", "player", "coach", "league", "score", "win"]
finance_words = ["market", "shares", "profit", "revenue", "investor", "bank", "trading"]
tech_words = ["software", "server", "cloud", "developer", "database", "release", "bug"]
filler = ["the", "a", "to", "of", "and", "in", "on", "for", "said", "new"]

def make_doc(topic_words, g, n_topic=8, n_fill=6):
    words = list(g.choice(topic_words, n_topic)) + list(g.choice(filler, n_fill))
    g.shuffle(words)
    return " ".join(words)

true_topic = rng.integers(0, 3, 300)
topic_pool = [sport_words, finance_words, tech_words]
docs_tm = [make_doc(topic_pool[t], rng) for t in true_topic]

tfidf_tm = TfidfVectorizer(stop_words=list(filler))
X_tfidf = tfidf_tm.fit_transform(docs_tm)
count_tm = CountVectorizer(stop_words=list(filler))
X_count = count_tm.fit_transform(docs_tm)

nmf = NMF(n_components=3, random_state=0, max_iter=500).fit(X_tfidf)
lda = LatentDirichletAllocation(n_components=3, random_state=0, max_iter=30).fit(X_count)

def show_topics(model, feature_names, n_top=8):
    for i, comp in enumerate(model.components_):
        top = np.argsort(comp)[::-1][:n_top]
        print(f"  topic {i}: {', '.join(feature_names[top])}")

print("NMF topics:")
show_topics(nmf, tfidf_tm.get_feature_names_out())
print("\nLDA topics:")
show_topics(lda, count_tm.get_feature_names_out())
print("\nBoth recover the three planted vocabularies without ever seeing the true_topic")
print("labels -- this is unsupervised structure discovery, exactly like clustering.")

In [ ]:
# Assign each document to its dominant topic and check against the (hidden) truth
doc_topic_nmf = nmf.transform(X_tfidf)
assigned = doc_topic_nmf.argmax(axis=1)

from sklearn.metrics import adjusted_rand_score
print(f"Adjusted Rand Index vs true topic labels: "
      f"{adjusted_rand_score(true_topic, assigned):.4f}  (1.0 = perfect, up to a relabelling)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
sns.heatmap(pd.crosstab(true_topic, assigned), annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_xlabel("NMF assigned topic"); ax[0].set_ylabel("true topic")
ax[0].set_title("Recovered vs true topic (as a contingency table)")

example = 5
ax[1].bar(range(3), doc_topic_nmf[example])
ax[1].set_xlabel("topic"); ax[1].set_ylabel("weight")
ax[1].set_title(f"Topic mixture for one document:\n'{docs_tm[example][:50]}...'")
plt.tight_layout(); plt.show()

In [ ]:
# Choosing k for topic models: reconstruction error, same spirit as the elbow method
ks = range(2, 9)
errs = [NMF(n_components=k, random_state=0, max_iter=400).fit(X_tfidf).reconstruction_err_
        for k in ks]
plt.plot(list(ks), errs, "o-", color="steelblue")
plt.axvline(3, color="crimson", ls="--", label="true number of topics")
plt.xlabel("number of topics"); plt.ylabel("NMF reconstruction error")
plt.title("An 'elbow' for topic count, exactly like K-Means"); plt.legend(fontsize=8)
plt.show()
print("As with clustering, there is no single correct k -- coherence, stability and")
print("above all human judgement ('do these topics make sense?') settle it in practice.")

---
## 9.7 A complete sentiment analysis pipeline

Putting it together: preprocessing, vectorisation, model, and honest evaluation with a train/
test split — precisely the workflow from statistics Notebook 10, applied to text.

In [ ]:
positive_words = ["excellent", "amazing", "great", "wonderful", "loved", "fantastic",
                  "perfect", "best", "delicious", "happy"]
negative_words = ["terrible", "awful", "worst", "disappointing", "bad", "horrible",
                  "waste", "poor", "broken", "rude"]
neutral_words = ["the", "product", "service", "food", "staff", "order", "delivery",
                 "price", "quality", "arrived", "was", "is", "a", "and"]
negators = ["not", "never", "hardly"]

def make_review(sentiment, g):
    if sentiment == 1:
        sentiment_words = list(g.choice(positive_words, g.integers(1, 3)))
    else:
        sentiment_words = list(g.choice(negative_words, g.integers(1, 3)))
    if g.random() < 0.15:                                  # occasional negation flips things
        sentiment_words = [rng.choice(negators)] + sentiment_words
    words = sentiment_words + list(g.choice(neutral_words, g.integers(4, 9)))
    g.shuffle(words)
    return " ".join(words)

m_r = 2_000
y_rev = (rng.random(m_r) < 0.5).astype(int)
X_rev = [make_review(s, rng) for s in y_rev]
print(f"{m_r} reviews, positive rate {y_rev.mean():.3f}\n")
for i in range(4):
    print(f"  [{'POS' if y_rev[i] else 'NEG'}] {X_rev[i]}")

In [ ]:
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_rev, y_rev, test_size=0.25, random_state=0,
                                              stratify=y_rev)

pipelines = {
    "CountVectorizer + NB": make_pipeline(CountVectorizer(), MultinomialNB()),
    "TF-IDF + Logistic": make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2),
                                       LogisticRegression(max_iter=2000)),
    "TF-IDF + NB": make_pipeline(TfidfVectorizer(), MultinomialNB()),
}
print(f"{'pipeline':<26}{'CV acc':>9}{'test acc':>10}{'test AUC':>10}")
for name, pipe in pipelines.items():
    cvv = cross_val_score(pipe, Xr_tr, yr_tr, cv=SKF).mean()
    pipe.fit(Xr_tr, yr_tr)
    pb = pipe.predict_proba(Xr_te)[:, 1]
    print(f"{name:<26}{cvv:>9.4f}{pipe.score(Xr_te, yr_te):>10.4f}"
          f"{roc_auc_score(yr_te, pb):>10.4f}")
print(f"\nBaseline: "
      f"{DummyClassifier(strategy='most_frequent').fit(Xr_tr, yr_tr).score(Xr_te, yr_te):.4f}")

In [ ]:
best = pipelines["TF-IDF + Logistic"]
pred = best.predict(Xr_te)
print(classification_report(yr_te, pred, target_names=["negative", "positive"]))

vec = best.named_steps["tfidfvectorizer"]
clf = best.named_steps["logisticregression"]
weights = pd.Series(clf.coef_[0], index=vec.get_feature_names_out())
top_pos = weights.sort_values(ascending=False).head(10)
top_neg = weights.sort_values().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
combined = pd.concat([top_neg, top_pos])
ax.barh(combined.index, combined.values,
        color=["crimson" if v < 0 else "steelblue" for v in combined.values])
ax.axvline(0, color="black", lw=1)
ax.set_title("Words that push sentiment negative (red) or positive (blue)")
plt.tight_layout(); plt.show()

test_reviews = ["not bad at all actually pretty good", "terrible service and awful food",
                "the delivery was fast and the staff were wonderful"]
for r in test_reviews:
    p = best.predict_proba([r])[0]
    print(f"\n'{r}'\n  P(positive) = {p[1]:.4f}  -> {'POSITIVE' if p[1]>0.5 else 'NEGATIVE'}")
print("\nThe negation example ('not bad') is the classic hard case for bag-of-words:")
print("bigrams give the model a fighting chance by capturing 'not bad' as one token.")

---
## 9.8 Where classical NLP ends and modern LLMs begin

| | Bag-of-words / TF-IDF | Word2vec / GloVe | Transformers (BERT, GPT) |
|---|---|---|---|
| Unit represented | Document | Word (one fixed vector) | Word **in context** (vector changes per sentence) |
| Word order | Ignored (or partial via n-grams) | Ignored within the window | Fully modelled |
| Polysemy ("bank") | N/A | **Cannot** distinguish senses | Distinguishes via context |
| Training data needed | None (counting) | Large corpus, unsupervised | Massive corpus, unsupervised pretraining |
| Compute | Trivial | Moderate | Very large |
| Interpretability | High | Low | Very low |

Everything in this notebook is still in daily production use — it is cheap, fast, needs no GPU,
and is often "good enough" or even the *right* choice when interpretability or latency matter
more than squeezing out the last few points of accuracy. Reach for a transformer when context
and word order carry the meaning (sarcasm, coreference, long-range dependencies) and you have
the data and compute budget to justify it.

In [ ]:
# The concrete limitation embeddings inherit but bag-of-words shares: polysemy
print("Toy corpus with 'bank' used in two unrelated senses:")
bank_corpus = [
    "i deposited money at the bank today",
    "the bank approved my loan application",
    "we sat by the river bank and watched the water",
    "the bank of the river was covered in reeds",
]
for i, s in enumerate(bank_corpus):
    print(f"  doc{i}: {s}")

X_bank = TfidfVectorizer().fit_transform(bank_corpus)
print("\nCosine similarity between the two FINANCIAL 'bank' docs (0 and 1):",
      f"{cosine_similarity(X_bank[0], X_bank[1])[0,0]:.4f}")
print("Cosine similarity between a FINANCIAL and a RIVER 'bank' doc (0 and 2):",
      f"{cosine_similarity(X_bank[0], X_bank[2])[0,0]:.4f}")
print("\nBag-of-words and classic word2vec both give 'bank' ONE vector regardless of sense.")
print("A transformer produces a DIFFERENT vector for 'bank' in each sentence, because its")
print("representation is computed from the surrounding words at inference time -- that is")
print("the single biggest conceptual leap from this notebook to modern NLP.")

---
## Exercises

**Exercise 1.** Build a preprocessing function that lowercases, removes punctuation, removes
stopwords, and applies the toy stemmer from section 9.1. Apply it to a short paragraph and
compare vocabulary size before and after.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
paragraph = ("The runners were training every morning. The coaches watched the players "
            "running drills. Teams that trained harder usually performed better in matches.")

def preprocess(text):
    tokens = re.findall(r"[a-z']+", text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [toy_stem(t) for t in tokens]
    return tokens

raw_tokens = re.findall(r"[a-z']+", paragraph.lower())
clean_tokens = preprocess(paragraph)

print(f"Raw tokens ({len(set(raw_tokens))} unique):")
print(f"  {sorted(set(raw_tokens))}")
print(f"\nPreprocessed tokens ({len(set(clean_tokens))} unique):")
print(f"  {sorted(set(clean_tokens))}")
print(f"\nVocabulary reduction: {len(set(raw_tokens))} -> {len(set(clean_tokens))} "
      f"({(1 - len(set(clean_tokens))/len(set(raw_tokens)))*100:.0f}% smaller)")
print("\n'runners'/'running' collapsed toward 'run*', 'coaches' lost its plural, and")
print("stopwords like 'the'/'were'/'that' disappeared entirely.")

**Exercise 2.** Compare unigrams, bigrams, and unigrams+bigrams as features for the sentiment
classifier from section 9.7. Report CV accuracy and vocabulary size for each, and identify the
single bigram with the strongest positive weight.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
print(f"{'ngram_range':>14}{'vocab size':>12}{'CV accuracy':>13}")
results = {}
for rng_ in [(1, 1), (2, 2), (1, 2)]:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=rng_, min_df=2),
                         LogisticRegression(max_iter=2000))
    vocab_size = len(TfidfVectorizer(ngram_range=rng_, min_df=2).fit(Xr_tr).vocabulary_)
    cvv = cross_val_score(pipe, Xr_tr, yr_tr, cv=SKF).mean()
    results[rng_] = pipe.fit(Xr_tr, yr_tr)
    print(f"{str(rng_):>14}{vocab_size:>12}{cvv:>13.4f}")

pipe_bi = results[(1, 2)]
vec2 = pipe_bi.named_steps["tfidfvectorizer"]
clf2 = pipe_bi.named_steps["logisticregression"]
w2 = pd.Series(clf2.coef_[0], index=vec2.get_feature_names_out())
bigram_weights = w2[[i for i in w2.index if " " in i]]
print(f"\nStrongest positive bigram: '{bigram_weights.idxmax()}' "
      f"(weight {bigram_weights.max():.3f})")
print(f"Strongest negative bigram: '{bigram_weights.idxmin()}' "
      f"(weight {bigram_weights.min():.3f})")
print("\nBigrams alone underperform unigrams (too sparse for this corpus size); combining")
print("both gives the model access to both single-word sentiment cues and negation phrases.")

**Exercise 3.** Fit topic models with $k = 2, 3, 4, 5$ topics on the sport/finance/tech corpus
from section 9.6, using both NMF and LDA. Compare how cleanly each recovers the known three
topics using the adjusted Rand index, and comment on why the two methods might disagree.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
rows3 = []
for k in (2, 3, 4, 5):
    nmf_k = NMF(n_components=k, random_state=0, max_iter=400).fit(X_tfidf)
    lda_k = LatentDirichletAllocation(n_components=k, random_state=0, max_iter=30).fit(X_count)
    nmf_assign = nmf_k.transform(X_tfidf).argmax(axis=1)
    lda_assign = lda_k.transform(X_count).argmax(axis=1)
    rows3.append({"k": k,
                  "NMF_ARI": adjusted_rand_score(true_topic, nmf_assign),
                  "LDA_ARI": adjusted_rand_score(true_topic, lda_assign)})
t3 = pd.DataFrame(rows3)
print(t3.round(4).to_string(index=False))

plt.plot(t3.k, t3.NMF_ARI, "o-", color="steelblue", label="NMF")
plt.plot(t3.k, t3.LDA_ARI, "o-", color="crimson", label="LDA")
plt.axvline(3, color="black", ls="--", label="true k")
plt.xlabel("k"); plt.ylabel("adjusted Rand index vs true topic")
plt.legend(fontsize=8); plt.title("Both methods peak near the true number of topics")
plt.show()

print("\nBoth peak at or near k=3, as expected. They can disagree because they optimise")
print("different objectives: NMF minimises TF-IDF reconstruction error (a geometric,")
print("deterministic factorisation), while LDA is a generative probabilistic model fit by")
print("an iterative approximate algorithm and is sensitive to its Dirichlet priors and")
print("random initialisation. On short, synthetic documents like these NMF is usually more")
print("stable; LDA tends to shine on longer, more naturally varied real text.")

**Exercise 4 (challenge).** Build a support-ticket router: classify tickets into
billing/technical/account categories using only 50 labelled examples per category. Compare
bag-of-words, TF-IDF with bigrams, and the averaged-embedding representation from section 9.5.
Report which representation wins under this small-data constraint and explain why.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
billing_w = ["invoice", "payment", "charge", "refund", "subscription", "billing", "price",
            "overcharged", "receipt", "renewal"]
technical_w = ["error", "crash", "bug", "login", "loading", "broken", "freeze", "sync",
              "update", "install"]
account_w = ["password", "profile", "email", "username", "settings", "verification",
            "locked", "access", "delete", "security"]
filler4 = ["the", "my", "is", "a", "and", "to", "for", "not", "please", "help"]
cat_words = {0: billing_w, 1: technical_w, 2: account_w}
cat_names = ["billing", "technical", "account"]

def make_ticket(cat, g):
    words = (list(g.choice(cat_words[cat], g.integers(2, 5)))
             + list(g.choice(filler4, g.integers(3, 7))))
    g.shuffle(words)
    return " ".join(words)

N_PER_CLASS = 50
y_tick = np.repeat([0, 1, 2], N_PER_CLASS)
X_tick = [make_ticket(c, rng) for c in y_tick]
Xt4_tr, Xt4_te, yt4_tr, yt4_te = train_test_split(X_tick, y_tick, test_size=0.3,
                                                  random_state=0, stratify=y_tick)
print(f"Training on {len(yt4_tr)} tickets ({N_PER_CLASS} per class before the split)\n")

# Build embeddings from the ticket vocabulary the same way as section 9.5
all_words = sorted(set(" ".join(X_tick).split()))
w2i_t = {w: i for i, w in enumerate(all_words)}
co_t = np.zeros((len(all_words), len(all_words)))
for doc in X_tick:
    toks = doc.split()
    for i, w in enumerate(toks):
        for j in range(max(0, i-2), min(len(toks), i+3)):
            if i != j:
                co_t[w2i_t[w], w2i_t[toks[j]]] += 1
emb_t = TruncatedSVD(n_components=20, random_state=0).fit_transform(co_t + 1e-6)

def embed_docs(docs):
    return np.array([doc_vector(d, emb_t, w2i_t) for d in docs])

X_emb_tr, X_emb_te = embed_docs(Xt4_tr), embed_docs(Xt4_te)

In [ ]:
print(f"{'representation':<30}{'CV accuracy':>13}{'test accuracy':>15}")

pipe_bow = make_pipeline(CountVectorizer(), LogisticRegression(max_iter=2000))
cv_bow = cross_val_score(pipe_bow, Xt4_tr, yt4_tr, cv=SKF).mean()
pipe_bow.fit(Xt4_tr, yt4_tr)
print(f"{'bag-of-words':<30}{cv_bow:>13.4f}{pipe_bow.score(Xt4_te, yt4_te):>15.4f}")

pipe_tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=1),
                           LogisticRegression(max_iter=2000))
cv_tfidf = cross_val_score(pipe_tfidf, Xt4_tr, yt4_tr, cv=SKF).mean()
pipe_tfidf.fit(Xt4_tr, yt4_tr)
print(f"{'TF-IDF + bigrams':<30}{cv_tfidf:>13.4f}{pipe_tfidf.score(Xt4_te, yt4_te):>15.4f}")

emb_clf = LogisticRegression(max_iter=2000).fit(X_emb_tr, yt4_tr)
cv_emb = cross_val_score(LogisticRegression(max_iter=2000), X_emb_tr, yt4_tr, cv=SKF).mean()
print(f"{'averaged embeddings (20d)':<30}{cv_emb:>13.4f}"
      f"{emb_clf.score(X_emb_te, yt4_te):>15.4f}")

print(f"\nBaseline (majority class): "
      f"{DummyClassifier(strategy='most_frequent').fit(Xt4_tr, yt4_tr).score(Xt4_te, yt4_te):.4f}")

In [ ]:
print("Which wins, and why:")
print("  Bag-of-words / TF-IDF tend to win at this scale, because the vocabulary is small")
print("  and DISTINCTIVE per class ('invoice' basically only appears in billing tickets),")
print("  so a sparse, high-dimensional, exact representation is easy for logistic")
print("  regression to separate with very little data.")
print()
print("  The averaged embeddings were themselves trained on only 150 short tickets, so they")
print("  are noisy and low-quality -- this is the opposite of the real-world case, where")
print("  embeddings are pretrained on billions of words and bring in OUTSIDE knowledge that")
print("  bag-of-words cannot access (e.g. that 'password' and 'credentials' are related,")
print("  even if 'credentials' never appeared in the labelled tickets).")
print()
print("  PRACTICAL RECOMMENDATION for a real 50-example-per-class router:")
print("  * start with TF-IDF + logistic regression or Naive Bayes: fast, robust, and this")
print("    experiment shows it is not disadvantaged by small labelled data")
print("  * if available, use PRETRAINED embeddings (word2vec/GloVe, or better, a sentence")
print("    transformer) rather than training your own on the tiny labelled set --")
print("    pretrained vectors carry semantic knowledge no 150-document corpus can supply")
print("  * as the labelled set grows into the thousands, revisit: transformer-based")
print("    embeddings typically overtake bag-of-words once there is enough data to fine-tune")

---
## Summary

| Concept | Key point |
|---|---|
| Tokenisation | Splitting text into units; deceptively non-trivial |
| Stemming vs lemmatisation | Rule-based chopping vs dictionary lookup; the latter is always a valid word |
| Bag-of-words | Word counts, order discarded |
| N-grams | Consecutive word sequences; recover local order and negation, at a vocabulary cost |
| TF-IDF | Downweights words common across documents, upweights distinctive ones |
| POS tagging / NER | Sequence labelling; classical methods use neighbouring tags as features |
| Distributional hypothesis | Words in similar contexts have similar meaning |
| word2vec / embeddings | Dense vectors where similarity is geometric; one vector per word (no context) |
| NMF / LDA | Unsupervised topic discovery; choose $k$ as with clustering |
| Transformers | Context-dependent vectors; solve polysemy that classical embeddings cannot |
| Practical rule | Bag-of-words/TF-IDF is a strong, cheap baseline — start there |

**Next up:** [Notebook 10 — Time Series Analytics](10.%20Time%20Series%20Analytics.ipynb),
where the ordering that bag-of-words throws away becomes the entire subject.